In [168]:
import pandas as pd
from sqlalchemy import create_engine
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import GridSearchCV, train_test_split

In [169]:
ind_df50 = pd.read_csv("/content/marathon_pred1.csv").iloc[:,1:]
ind_df50.head()

,event_date,host_country,event_name,event_distance/length,athlete_id,athlete_country,event_number_of_finishers,athlete_age,athlete_gender,athlete_performance(h),athlete_average_speed,age_category,year,position,medal
0,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37542,IND,160,32,M,3.38,13.700,Prime Age Athletes,2018,1,Gold
1,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37543,IND,160,26,M,3.55,12.762,Prime Age Athletes,2018,2,Silver
2,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37544,IND,160,24,M,4.02,12.357,Youth\Young Athlete,2018,3,Bronze
3,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37545,IND,160,43,M,4.04,12.271,Middle-Aged Athletes,2018,4,No Medal
4,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37546,IND,160,34,M,4.11,11.950,Prime Age Athletes,2018,5,No Medal


In [170]:
ind_df50['Yes/No'] = ind_df50.apply(lambda row: 'Yes' if row['medal'] in ('Gold','Silver','Bronze') else 'No',axis= 1)
ind_df50

,event_date,host_country,event_name,event_distance/length,athlete_id,athlete_country,event_number_of_finishers,athlete_age,athlete_gender,athlete_performance(h),athlete_average_speed,age_category,year,position,medal,Yes/No
0,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37542,IND,160,32,M,3.38,13.700,Prime Age Athletes,2018,1,Gold,Yes
1,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37543,IND,160,26,M,3.55,12.762,Prime Age Athletes,2018,2,Silver,Yes
2,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37544,IND,160,24,M,4.02,12.357,Youth\Young Athlete,2018,3,Bronze,Yes
3,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37545,IND,160,43,M,4.04,12.271,Middle-Aged Athletes,2018,4,No Medal,No
4,2018-02-25,IND,Tata Ultra Marathon (IND)-2018,50km,37546,IND,160,34,M,4.11,11.950,Prime Age Athletes,2018,5,No Medal,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6184,2015-01-11,IND,Vadodara Ultra Trail (IND)-2015,55km,387505,IND,44,33,M,11.17,4.873,Prime Age Athletes,2015,36,No Medal,No
6185,2015-01-11,IND,Vadodara Ultra Trail (IND)-2015,55km,1516068,IND,44,36,M,11.17,4.873,Semi-Middel Age Athletes,2015,37,No Medal,No
6186,2015-01-11,IND,Vadodara Ultra Trail (IND)-2015,55km,388056,IND,44,42,M,11.17,4.873,Middle-Aged Athletes,2015,38,No Medal,No
6187,2015-01-11,IND,Vadodara Ultra Trail (IND)-2015,55km,264677,IND,44,36,M,11.17,4.872,Semi-Middel Age Athletes,2015,39,No Medal,No


In [171]:
ind_df50.columns

Index(['event_date', 'host_country', 'event_name', 'event_distance/length',
       'athlete_id', 'athlete_country', 'event_number_of_finishers',
       'athlete_age', 'athlete_gender', 'athlete_performance(h)',
       'athlete_average_speed', 'age_category', 'year', 'position', 'medal',
       'Yes/No'],
      dtype='object')

In [172]:
data = ind_df50[['event_distance/length','event_number_of_finishers','athlete_country','athlete_age','athlete_gender','athlete_average_speed','athlete_performance(h)','Yes/No']]
data

,event_distance/length,event_number_of_finishers,athlete_country,athlete_age,athlete_gender,athlete_average_speed,athlete_performance(h),Yes/No
0,50km,160,IND,32,M,13.700,3.38,Yes
1,50km,160,IND,26,M,12.762,3.55,Yes
2,50km,160,IND,24,M,12.357,4.02,Yes
3,50km,160,IND,43,M,12.271,4.04,No
4,50km,160,IND,34,M,11.950,4.11,No
...,...,...,...,...,...,...,...,...
6184,55km,44,IND,33,M,4.873,11.17,No
6185,55km,44,IND,36,M,4.873,11.17,No
6186,55km,44,IND,42,M,4.873,11.17,No
6187,55km,44,IND,36,M,4.872,11.17,No


In [173]:
data.isna().sum()

,0
event_distance/length,0
event_number_of_finishers,0
athlete_country,0
athlete_age,0
athlete_gender,0
athlete_average_speed,0
athlete_performance(h),0
Yes/No,0


In [174]:
data.dtypes

,0
event_distance/length,object
event_number_of_finishers,int64
athlete_country,object
athlete_age,int64
athlete_gender,object
athlete_average_speed,float64
athlete_performance(h),float64
Yes/No,object


In [175]:
df = data

In [176]:
# One Hot Encoding
# 'event_distance/length','athlete_country','athlete_gender'

dist_dummies = pd.get_dummies(df['event_distance/length'],dtype= int)
df = pd.concat([df,dist_dummies],axis= 1)
df.drop(columns= ['event_distance/length','56km'],inplace= True)

count_dummies = pd.get_dummies(df['athlete_country'],dtype= int)
df = pd.concat([df,count_dummies],axis= 1)
df.drop(columns= ['athlete_country','XXX'],inplace= True)

gender_dummies = pd.get_dummies(df['athlete_gender'],dtype= int)
df = pd.concat([df,gender_dummies],axis= 1)
df.drop(columns= ['athlete_gender'],inplace= True)


In [177]:
le = LabelEncoder()

df['Yes/No'] = le.fit_transform(df['Yes/No'])

In [179]:
X = df.drop(columns= 'Yes/No')
y = df[['Yes/No']]

In [180]:
X_train,X_test,y_train,y_test = train_test_split(X, y, train_size= 0.7,random_state= 42)

In [181]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

oversample = SMOTE(sampling_strategy = "auto",random_state= 42)
X_train, y_train = oversample.fit_resample(X_train, y_train)

In [182]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


pipeline = Pipeline([
    ('logreg', LogisticRegression())
])


param_grid = {
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__penalty': ['l1', 'l2'],
    'logreg__solver': ['liblinear', 'saga'],
    'logreg__max_iter': [100, 200, 300],
    'logreg__class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 120 candidates, totalling 600 fits


/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GridSearchCV(cv=5, estimator=Pipeline(steps=[('logreg', LogisticRegression())]),
             n_jobs=-1,
             param_grid={'logreg__C': [0.01, 0.1, 1, 10, 100],
                         'logreg__class_weight': [None, 'balanced'],
                         'logreg__max_iter': [100, 200, 300],
                         'logreg__penalty': ['l1', 'l2'],
                         'logreg__solver': ['liblinear', 'saga']},
             scoring='accuracy', verbose=2)

In [189]:
model = LogisticRegression(C=100, max_iter=200, penalty='l1', solver='liblinear')

In [190]:
model.fit(X_train,y_train)

/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression(C=100, max_iter=200, penalty='l1', solver='liblinear')

In [191]:
model.score(X_train,y_train)

0.8987189751801441

In [192]:
model.score(X_test,y_test)

0.8901453957996769

In [193]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

# Predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

Model Accuracy: 0.89


In [194]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.96      0.91      0.94      1619
           1       0.55      0.74      0.63       238

    accuracy                           0.89      1857
   macro avg       0.76      0.83      0.78      1857
weighted avg       0.91      0.89      0.90      1857

